# 07 — Adaptive Simulation Depth Engine

**Adaptive Multiscale Biological Simulation AI**

Phase 2: The core research innovation — an engine that dynamically decides where to apply expensive physics.

**The problem**: Simulating every atom at quantum precision is impossible for biological systems (~10^23 atoms). We need to **allocate precision dynamically** — high detail where activity is high, coarse elsewhere.

This notebook covers:
1. The depth hierarchy (5 levels)
2. Importance scoring (when to zoom in)
3. Depth selector (AI-based depth prediction)
4. Switchoff engine (the adaptive loop)
5. Loss function with physics constraints
6. Visualization of region allocation

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.insert(0, str(Path(__file__).parent.parent.parent))
from src.sim.deepness.simulation_depth import SimulationDepth, DepthSelector
from src.sim.layers.depth_selector import DepthSelectorLayer
from src.sim.layers.uncertainty_estimator import UncertaintyEstimator, ImportanceScorer

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 13})

device = torch.device('cpu')
print(f'Using: {device}')
print(f'Available depth levels:')
for k, v in SimulationDepth.NAMES.items():
    print(f'  {k}: {v}')

# Create a simulated molecule for demonstration
n_atoms = 50
atomic_numbers = [1]*10 + [6]*15 + [7]*10 + [8]*10 + [9]*5  # H, C, N, O, F
positions = np.random.uniform(-2, 2, (n_atoms, 3))
charges = np.random.normal(0, 0.1, n_atoms)

## 1. The Depth Hierarchy

| Depth | Resolution | When to Use |
|---|----|------|------|
| 0 | Statistical | Low activity regions, distant particles |
| 1 | Cellular | Medium activity, diffusion regimes |
| 2 | Molecular | Standard MD, molecular interactions |
| 3 | Atomistic Fine | Bond formation/breaking, reactive regions |
| 4 | Quantum | Electron transfer, active sites, bonds |

In [ ]:
# Visualize depth hierarchy
fig, ax = plt.subplots(figsize=(10, 6))

# Create depth levels
depths = [SimulationDepth.STATISTICAL, SimulationDepth.CELLULAR, 
          SimulationDepth.MOLECULAR, SimulationDepth.ATOMISTIC_FINE, 
          SimulationDepth.QUANTUM]
names = [SimulationDepth.NAMES[d] for d in depths]

# Colors for each depth
colors = ['lightgray', 'lightblue', 'lightgreen', 'orange', 'red']

bars = ax.barh(range(len(depths)), [100, 80, 60, 40, 20], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_yticks(range(len(depths)))
ax.set_yticklabels([f"{d}: {names[i]}" for i, d in enumerate(depths)])
ax.set_xlabel('Compute Cost (relative)')
ax.set_title('Simulation Depth Hierarchy')
ax.set_xlim(0, 120)
ax.grid(True, alpha=0.3)

# Add descriptions
descriptions = [
    "Coarse-grained ensemble statistics",
    "Mesoscopic molecular field",
    "Standard MD simulation",
    "High-res MD with tight cutoffs",
    "DFT quantum chemistry"
]
for i, (bar, desc) in enumerate(zip(bars, descriptions)):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2, 
           desc, va='center', fontsize=10)

plt.tight_layout()
plt.savefig('figs/07_depth_hierarchy.png', bbox_inches='tight')
plt.show()
print('Depth hierarchy visualization saved')

## 2. Importance Scoring

The engine predicts **depth per atom** using multiple signals:

**Signal 1: Force Magnitude**
High force → high activity → need deeper simulation

**Signal 2: Energy Gradient**
Large gradient → near chemical potential → need quantum

**Signal 3: Uncertainty**
High prediction uncertainty → needs validation

**Signal 4: Charge Transfer**
Charged atoms → active sites → quantum region

**Signal 5: Local Density**
High density → more interactions → need accuracy

In [ ]:
# Demonstrate importance scoring
depth_selector = ImportanceScorer(hidden_dim=16)

# Simulate fake signals
force_mag = torch.tensor([0.1, 0.3, 0.8, 0.2, 1.2, 0.05, 0.5, 0.9, 0.3, 0.15]).unsqueeze(-1)
energy_grad = torch.tensor([0.1, 0.4, 0.7, 0.2, 1.1, 0.05, 0.4, 0.8, 0.3, 0.1]).unsqueeze(-1)
uncertainty = torch.tensor([0.1, 0.3, 0.6, 0.2, 0.9, 0.05, 0.4, 0.7, 0.3, 0.1]).unsqueeze(-1)
local_density = torch.tensor([0.2, 0.4, 0.6, 0.3, 0.8, 0.1, 0.5, 0.7, 0.3, 0.15]).unsqueeze(-1)
charge_mag = torch.tensor([0.05, 0.2, 0.5, 0.1, 0.8, 0.02, 0.4, 0.6, 0.2, 0.08]).unsqueeze(-1)

importance = depth_selector(force_mag, energy_grad, uncertainty, local_density, charge_mag)

# Convert to discrete depth
thresholds = torch.tensor([0.2, 0.4, 0.6, 0.8])
depths = torch.zeros(len(importance), dtype=torch.long)
for i, thresh in enumerate(thresholds):
    if importance.any() > thresh:
        depths[importance.squeeze() > thresh] = i + 1

print('Importance scores and depth assignments:')
for i in range(len(importance)):
    print(f"  Atom {i}: force={force_mag[i]:.2f}, grad={energy_grad[i]:.2f}, "
          f"unc={uncertainty[i]:.2f}, imp={importance[i]:.2f}, depth={depths[i]} "
          f"({SimulationDepth.NAMES[depths[i].item()]})")

# Visualize
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(importance))
width = 0.35
bars1 = ax.bar(x - width/2, importance.numpy(), width, label='Importance', color='steelblue')
bars2 = ax.bar(x + width/2, [SimulationDepth.NAMES[d.item()] for d in depths], width, label='Depth', color='coral')
ax.set_xlabel('Atom Index')
ax.set_ylabel('Value')
ax.set_title('Importance Scoring → Depth Assignment')
ax.set_xticks(x)
ax.set_xticklabels([f"A{i}" for i in range(len(importance))])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figs/07_importance_scoring.png', bbox_inches='tight')
plt.show()
print('Importance scoring visualization saved')

## 3. AI-Based Depth Selector

The MACE model's node features can be directly used to predict depth:

In [ ]:
# Create a depth selector layer
hidden_dim = 32
n_atoms = 10
node_features = torch.randn(n_atoms, hidden_dim)

# Initialize depth selector
sel = DepthSelectorLayer(hidden_dim=hidden_dim, n_depths=5)

# Get depth scores
depth_scores, depth_probs = sel(node_features)

print(f"Node features shape: {node_features.shape}")
print(f"Depth scores: {depth_scores.tolist()}")
print(f"Depth probabilities shape: {depth_probs.shape}")
print(f"\nDepth probabilities:")
for i in range(n_atoms):
    probs = depth_probs[i].tolist()
    depths_str = ', '.join([f"{SimulationDepth.NAMES[int(d)]:10s}: {p:.3f}" for d, p in enumerate(probs)])
    print(f"  Atom {i}: {depths_str}")
    
# Visualize
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(n_atoms)
for i, layer_name in enumerate([SimulationDepth.NAMES[d] for d in range(5)]):
    ax.plot(x, depth_probs[:, i].numpy(), 'o-', label=layer_name, alpha=0.7)
ax.set_xlabel('Atom Index')
ax.set_ylabel('Probability')
ax.set_title('Depth Selection Probabilities per Atom')
ax.set_xticks(x)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figs/07_depth_probabilities.png', bbox_inches='tight')
plt.show()
print('Depth probabilities visualization saved')

## 4. Switchoff Engine — The Adaptive Loop

In [ ]:
class SwitchoffEngine:
    """Engine that switches simulation depth dynamically."""
    
    def __init__(self, force_threshold=0.5, gradient_threshold=1.0):
        self.force_threshold = force_threshold
        self.gradient_threshold = gradient_threshold
        self.depth_selector = DepthSelectorLayer(hidden_dim=32)
        self.depth_map = {}
        self.time_step = 0
    
    def update_depth(self, forces, energies, atoms):
        """Update the depth map based on current simulation state."""
        for atom in atoms:
            depth = self._determine_depth(atom, forces[atom], energies[atom])
            self.depth_map[atom] = depth
        self.time_step += 1
        return self.depth_map.copy()
    
    def _determine_depth(self, atom, force, energy):
        if force > self.force_threshold:
            if energy > self.gradient_threshold:
                return 4  # Quantum
            else:
                return 3  # Atomistic fine
        elif energy > self.gradient_threshold:
            return 2  # Molecular
        elif force > self.force_threshold * 0.5:
            return 1  # Cellular
        else:
            return 0  # Statistical
    
    def get_region_stats(self):
        counts = {d: 0 for d in range(5)}
        for depth in self.depth_map.values():
            counts[depth] = counts.get(depth, 0) + 1
        return counts

# Demonstrate
engine = SwitchoffEngine(force_threshold=0.5, gradient_threshold=1.0)
atoms = [f"atom_{i}" for i in range(10)]
forces = [0.1, 0.3, 0.8, 0.2, 1.2, 0.05, 0.5, 0.9, 0.3, 0.15]
energies = [0.1, 0.4, 0.7, 0.2, 1.1, 0.05, 0.4, 0.8, 0.3, 0.1]

depth_map = engine.update_depth(forces, energies, atoms)

print('Depth mapping:')
for atom, depth in depth_map.items():
    print(f"  {atom}: depth {depth} ({SimulationDepth.NAMES[depth]})")

print(f"\nRegion stats: {engine.get_region_stats()}")

## 5. Full Simulation Loop with Adaptive Depth

Demonstrate the full adaptive simulation workflow:

In [ ]:
# Simulate adaptive behavior over time
n_timesteps = 100
region_counts = np.zeros((n_timesteps, 5))

for t in range(n_timesteps):
    # Simulate changing activity over time
    forces = np.array([0.1 * np.cos(2*np.pi*t/10), 
                      0.3 + 0.2*np.sin(t/5),
                      0.5 + 0.3*np.cos(t/3),
                      0.2 * np.sin(t/4),
                      1.0 + 0.5*np.cos(t/2)])
    
    depths = torch.zeros(5, dtype=torch.long)
    for i in range(5):
        if forces[i] > 0.8:
            depths[i] = 4
        elif forces[i] > 0.5:
            depths[i] = 3
        elif forces[i] > 0.3:
            depths[i] = 2
        elif forces[i] > 0.15:
            depths[i] = 1
        else:
            depths[i] = 0
    
    counts = {d: 0 for d in range(5)}
    for d in depths:
        counts[d.item()] = counts.get(d.item(), 0) + 1
    
    region_counts[t] = [counts.get(d, 0) for d in range(5)]

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
timesteps = np.arange(n_timesteps)
colors = ['gray', 'lightblue', 'lightgreen', 'orange', 'red']
layer_names = [SimulationDepth.NAMES[d] for d in range(5)]

ax.stackplot(timesteps, region_counts.T, 
            labels=layer_names, colors=colors[:5])
ax.set_xlabel('Timestep')
ax.set_ylabel('Number of regions')
ax.set_title('Adaptive Depth Allocation Over Time')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figs/07_adaptive_time.png', bbox_inches='tight')
plt.show()
print('Adaptive allocation visualization saved')

## 6. Loss Function with Physics Constraints

The adaptive simulation engine uses a multi-component loss:

L_total = L_data + L_physics + L_energy + L_entropy + L_symmetry

Where:
- L_data: Force/energy MSE
- L_physics: Conservation constraints
- L_energy: Energy conservation
- L_entropy: Thermostat target
- L_symmetry: Equivariance constraint

In [ ]:
# Define the loss
class AdaptiveLoss(torch.nn.Module):
    def __init__(self, w_data=1.0, w_physics=1.0, w_energy=1.0, 
                w_entropy=0.5, w_symmetry=0.1):
        super().__init__()
        self.w_data = w_data
        self.w_physics = w_physics
        self.w_energy = w_energy
        self.w_entropy = w_entropy
        self.w_symmetry = w_symmetry
    
    def forward(self, pred_force, true_force, pred_energy, true_energy,
               pred_temp, target_temp, equivariance_violation):
        data_loss = (pred_force - true_force).pow(2).mean() + (pred_energy - true_energy).pow(2).mean()
        physics_loss = torch.abs(pred_force).sum()
        energy_conserv = (pred_energy - true_energy).abs().sum()
        entropy_loss = (pred_temp - target_temp).abs()
        symmetry_loss = equivariance_violation
        
        return (self.w_data * data_loss
               + self.w_physics * physics_loss
               + self.w_energy * energy_conserv
               + self.w_entropy * entropy_loss
               + self.w_symmetry * symmetry_loss)

print("Adaptive loss function defined.")
print("L_total = w_data*L_data + w_phys*L_phys + w_energy*L_e + w_entropy*L_ent + w_symm*L_sym")
print("\nTypical weights:")
print("  w_data=1.0  (force/energy MSE)")
print("  w_physics=1.0  (momentum conservation)")
print("  w_energy=1.0  (energy conservation)")
print("  w_entropy=0.5 (temperature target)")
print("  w_symmetry=0.1 (equivariance violation)")

## Summary

| Component | Description |
|------|----|------|
| **Depth Hierarchy** | 5 levels from statistical to quantum |
| **Importance Scorer** | Multi-signal scoring system |
| **Depth Selector Layer** | AI-based depth prediction |
| **Switchoff Engine** | Adaptive simulation controller |
| **Adaptive Loss** | Multi-component physics constraints |

**Key insight**: The engine **dynamically allocates** simulation depth — not per-molecule, but **per-atom**, adapting in real-time.

**Next:** `08_mvp_v1.ipynb` — MVP v1 implementation: adaptive simulation depth with AI prediction.

# Architecture Summary:
1. Input: atoms, positions, forces
2. Signal computation: force, gradient, uncertainty, density
3. Depth selection: importance scorer → discrete depths
4. Simulation: different depth for each atom
5. Loss: multi-component physics constraints
6. Output: depth map + simulation results